# 1. Data Collection

This notebook collects a **four-year hourly dataset (2022-09-01 to 2026-08-31)** for Hanoi using:

- **Open-Meteo Air Quality API** for PM2.5, PM10, and NO2;
- **Open-Meteo Historical Weather API** for meteorological predictors.

The requested coordinate: **21.0052, 105.8418**.

Both APIs are requested in **`Asia/Ho_Chi_Minh` local time (UTC+7)** so the hourly timestamps have an immediate local interpretation. A UTC copy is also retained for auditing and portability.

> Important: Open-Meteo air-quality values are gridded/model-based values (CAMS), not direct measurements from the former OpenAQ ground sensor. The API may return a grid-cell centre that differs from the requested coordinate.

In [1]:
from pathlib import Path
import json
import requests
import pandas as pd
import numpy as np

## Study configuration

In [2]:
LATITUDE = 21.0052
LONGITUDE = 105.8418

START_DATE = "2022-09-01"
END_DATE = "2026-08-31"   # inclusive
LOCAL_TZ = "Asia/Ho_Chi_Minh"

AIR_QUALITY_VARIABLES = [
    "pm2_5",
    "pm10",
    "nitrogen_dioxide",
]

WEATHER_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
]

RAW_DIR = Path("../data/raw/openmeteo")
INTERIM_DIR = Path("../data/interim")

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Collect historical air-quality data

In [3]:
AIR_QUALITY_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

def get_air_quality(
    latitude,
    longitude,
    start_date,
    end_date,
    hourly_variables,
    timezone_name=LOCAL_TZ,
):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": timezone_name,
        "domains": "cams_global",
        "cell_selection": "nearest",
    }

    response = requests.get(
        AIR_QUALITY_URL,
        params=params,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

In [4]:
air_quality_response = get_air_quality(
    latitude=LATITUDE,
    longitude=LONGITUDE,
    start_date=START_DATE,
    end_date=END_DATE,
    hourly_variables=AIR_QUALITY_VARIABLES,
)

print("Returned AQ grid centre:",
      air_quality_response["latitude"],
      air_quality_response["longitude"])
print("Returned timezone:", air_quality_response["timezone"])
print("UTC offset (seconds):", air_quality_response["utc_offset_seconds"])

Returned AQ grid centre: 21.0 105.80002
Returned timezone: Asia/Ho_Chi_Minh
UTC offset (seconds): 25200


## Convert air-quality response to DataFrame

In [5]:
df_air_quality = pd.DataFrame(
    air_quality_response["hourly"]
).rename(columns={
    "time": "timestamp_local",
    "pm2_5": "pm25",
    "nitrogen_dioxide": "no2",
})

# Open-Meteo returns local clock timestamps when timezone is specified.
df_air_quality["timestamp_local"] = (
    pd.to_datetime(df_air_quality["timestamp_local"])
      .dt.tz_localize(LOCAL_TZ)
)

# Keep a UTC copy for auditing / interoperability.
df_air_quality["timestamp_utc"] = (
    df_air_quality["timestamp_local"]
      .dt.tz_convert("UTC")
)

df_air_quality.head()

,timestamp_local,pm25,pm10,no2,timestamp_utc
0,2022-09-01 00:00:00+07:00,71.5,103.1,29.1,2022-08-31 17:00:00+00:00
1,2022-09-01 01:00:00+07:00,62.4,89.9,25.5,2022-08-31 18:00:00+00:00
2,2022-09-01 02:00:00+07:00,49.2,71.0,21.8,2022-08-31 19:00:00+00:00
3,2022-09-01 03:00:00+07:00,36.6,52.9,17.6,2022-08-31 20:00:00+00:00
4,2022-09-01 04:00:00+07:00,28.7,41.6,13.2,2022-08-31 21:00:00+00:00


## Validate air-quality coverage

In [6]:
expected_local = pd.date_range(
    start=f"{START_DATE} 00:00:00",
    end=f"{END_DATE} 23:00:00",
    freq="h",
    tz=LOCAL_TZ,
)

print("Expected hours:", len(expected_local))
print("AQ rows:", len(df_air_quality))
print("Duplicate local timestamps:",
      df_air_quality["timestamp_local"].duplicated().sum())
print("Start:", df_air_quality["timestamp_local"].min())
print("End:", df_air_quality["timestamp_local"].max())

aq_missing_summary = pd.DataFrame({
    "missing_count": df_air_quality[
        ["pm25", "pm10", "no2"]
    ].isna().sum(),
    "missing_pct": (
        df_air_quality[["pm25", "pm10", "no2"]]
        .isna()
        .mean()
        .mul(100)
        .round(3)
    ),
})

aq_missing_summary

Expected hours: 35064
AQ rows: 35064
Duplicate local timestamps: 0
Start: 2022-09-01 00:00:00+07:00
End: 2026-08-31 23:00:00+07:00


,missing_count,missing_pct
pm25,0,0.0
pm10,0,0.0
no2,0,0.0


In [7]:
# Save the raw hourly table returned by the API.
df_air_quality.to_csv(
    RAW_DIR / "hanoi_air_quality_2022-09-01_2026-08-31.csv",
    index=False,
)

# Step 2: Inspect temporal continuity of the air-quality series

In [8]:
aq_time_diff = (
    df_air_quality
    .sort_values("timestamp_local")["timestamp_local"]
    .diff()
)

print("Most common time differences:")
display(aq_time_diff.value_counts().head())

print(
    "Number of observed timestamp gaps > 1 hour:",
    int((aq_time_diff > pd.Timedelta(hours=1)).sum()),
)

Most common time differences:


timestamp_local
0 days 01:00:00    35063
Name: count, dtype: int64

Number of observed timestamp gaps > 1 hour: 0


/var/folders/sb/pmlt6jxs48zgnsjrc491m_s80000gn/T/ipykernel_86498/652125346.py:12: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  int((aq_time_diff > pd.Timedelta(hours=1)).sum()),


# Step 3: Collect historical weather data

In [9]:
HISTORICAL_WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

def get_historical_weather(
    latitude,
    longitude,
    start_date,
    end_date,
    hourly_variables,
    timezone_name=LOCAL_TZ,
):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": timezone_name,
        "wind_speed_unit": "ms",
        "precipitation_unit": "mm",
        "temperature_unit": "celsius",
        "cell_selection": "land",
    }

    response = requests.get(
        HISTORICAL_WEATHER_URL,
        params=params,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

In [10]:
weather_response = get_historical_weather(
    latitude=LATITUDE,
    longitude=LONGITUDE,
    start_date=START_DATE,
    end_date=END_DATE,
    hourly_variables=WEATHER_VARIABLES,
)

print("Returned weather grid centre:",
      weather_response["latitude"],
      weather_response["longitude"])
print("Returned timezone:", weather_response["timezone"])
print("UTC offset (seconds):", weather_response["utc_offset_seconds"])

Returned weather grid centre: 20.984182 105.882355
Returned timezone: Asia/Ho_Chi_Minh
UTC offset (seconds): 25200


## Convert weather response to DataFrame

In [11]:
df_weather = pd.DataFrame(
    weather_response["hourly"]
).rename(columns={
    "time": "timestamp_local"
})

df_weather["timestamp_local"] = (
    pd.to_datetime(df_weather["timestamp_local"])
      .dt.tz_localize(LOCAL_TZ)
)

df_weather["timestamp_utc"] = (
    df_weather["timestamp_local"]
      .dt.tz_convert("UTC")
)

df_weather.head()

,timestamp_local,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,timestamp_utc
0,2022-09-01 00:00:00+07:00,27.6,91,0.0,1006.6,1.70,183,2022-08-31 17:00:00+00:00
1,2022-09-01 01:00:00+07:00,27.3,93,0.0,1006.2,0.99,135,2022-08-31 18:00:00+00:00
2,2022-09-01 02:00:00+07:00,25.7,94,0.5,1006.7,5.59,29,2022-08-31 19:00:00+00:00
3,2022-09-01 03:00:00+07:00,24.7,93,3.9,1006.8,4.64,53,2022-08-31 20:00:00+00:00
4,2022-09-01 04:00:00+07:00,24.5,96,0.9,1007.3,4.57,29,2022-08-31 21:00:00+00:00


## Validate weather data

In [12]:
print("Weather rows:", len(df_weather))
print(
    "Duplicate local timestamps:",
    df_weather["timestamp_local"].duplicated().sum(),
)
print("Start:", df_weather["timestamp_local"].min())
print("End:", df_weather["timestamp_local"].max())

weather_missing_summary = pd.DataFrame({
    "missing_count": df_weather[WEATHER_VARIABLES].isna().sum(),
    "missing_pct": (
        df_weather[WEATHER_VARIABLES]
        .isna()
        .mean()
        .mul(100)
        .round(3)
    ),
})

weather_missing_summary

Weather rows: 35064
Duplicate local timestamps: 0
Start: 2022-09-01 00:00:00+07:00
End: 2026-08-31 23:00:00+07:00


,missing_count,missing_pct
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
precipitation,0,0.0
surface_pressure,0,0.0
wind_speed_10m,0,0.0
wind_direction_10m,0,0.0


In [13]:
df_weather.to_csv(
    RAW_DIR / "hanoi_historical_weather_2022-09-01_2026-08-31.csv",
    index=False,
)

# Step 4: Align air quality and weather on the complete local hourly timeline

A canonical local-time hourly index is created first. Both API tables are then left-joined to this timeline. This makes any missing hour explicit rather than silently dropping it during an inner join.

In [14]:
timeline = pd.DataFrame({
    "timestamp_local": expected_local
})

timeline["timestamp_utc"] = (
    timeline["timestamp_local"]
    .dt.tz_convert("UTC")
)

aq_model = df_air_quality[
    ["timestamp_local", "pm25", "pm10", "no2"]
].copy()

weather_model = df_weather[
    ["timestamp_local"] + WEATHER_VARIABLES
].copy()

df_merged = (
    timeline
    .merge(
        aq_model,
        on="timestamp_local",
        how="left",
        validate="one_to_one",
    )
    .merge(
        weather_model,
        on="timestamp_local",
        how="left",
        validate="one_to_one",
    )
    .sort_values("timestamp_local")
    .reset_index(drop=True)
)

df_merged.head()

,timestamp_local,timestamp_utc,pm25,pm10,no2,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m
0,2022-09-01 00:00:00+07:00,2022-08-31 17:00:00+00:00,71.5,103.1,29.1,27.6,91,0.0,1006.6,1.70,183
1,2022-09-01 01:00:00+07:00,2022-08-31 18:00:00+00:00,62.4,89.9,25.5,27.3,93,0.0,1006.2,0.99,135
2,2022-09-01 02:00:00+07:00,2022-08-31 19:00:00+00:00,49.2,71.0,21.8,25.7,94,0.5,1006.7,5.59,29
3,2022-09-01 03:00:00+07:00,2022-08-31 20:00:00+00:00,36.6,52.9,17.6,24.7,93,3.9,1006.8,4.64,53
4,2022-09-01 04:00:00+07:00,2022-08-31 21:00:00+00:00,28.7,41.6,13.2,24.5,96,0.9,1007.3,4.57,29


## Validate integrated dataset

In [15]:
print("Shape:", df_merged.shape)
print("Expected hours:", len(expected_local))
print(
    "Duplicate local timestamps:",
    df_merged["timestamp_local"].duplicated().sum(),
)

assert len(df_merged) == len(expected_local)
assert df_merged["timestamp_local"].is_monotonic_increasing
assert not df_merged["timestamp_local"].duplicated().any()

merged_missing_summary = pd.DataFrame({
    "missing_count": df_merged.isna().sum(),
    "missing_pct": (
        df_merged.isna()
        .mean()
        .mul(100)
        .round(3)
    ),
})

merged_missing_summary

Shape: (35064, 11)
Expected hours: 35064
Duplicate local timestamps: 0


,missing_count,missing_pct
timestamp_local,0,0.0
timestamp_utc,0,0.0
pm25,0,0.0
pm10,0,0.0
no2,0,0.0
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
precipitation,0,0.0
surface_pressure,0,0.0
wind_speed_10m,0,0.0


## Source/grid metadata

The requested coordinate is identical for both APIs, but the returned AQ and weather grid-cell centres may differ because the two products use different underlying grids. Store those coordinates explicitly so this limitation is documented and reproducible.

In [16]:
source_metadata = {
    "requested_coordinate": {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
    },
    "study_period": {
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
    "timezone": LOCAL_TZ,
    "air_quality": {
        "source": "Open-Meteo Air Quality API / CAMS Global",
        "grid_latitude": air_quality_response["latitude"],
        "grid_longitude": air_quality_response["longitude"],
        "utc_offset_seconds": air_quality_response["utc_offset_seconds"],
        "variables": AIR_QUALITY_VARIABLES,
    },
    "weather": {
        "source": "Open-Meteo Historical Weather API",
        "grid_latitude": weather_response["latitude"],
        "grid_longitude": weather_response["longitude"],
        "utc_offset_seconds": weather_response["utc_offset_seconds"],
        "variables": WEATHER_VARIABLES,
    },
}

with open(
    RAW_DIR / "source_metadata_2022-09-01_2026-08-31.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(source_metadata, f, ensure_ascii=False, indent=2)

source_metadata

{'requested_coordinate': {'latitude': 21.0052, 'longitude': 105.8418},
 'study_period': {'start_date': '2022-09-01', 'end_date': '2026-08-31'},
 'timezone': 'Asia/Ho_Chi_Minh',
 'air_quality': {'source': 'Open-Meteo Air Quality API / CAMS Global',
  'grid_latitude': 21.0,
  'grid_longitude': 105.80002,
  'utc_offset_seconds': 25200,
  'variables': ['pm2_5', 'pm10', 'nitrogen_dioxide']},
 'weather': {'source': 'Open-Meteo Historical Weather API',
  'grid_latitude': 20.984182,
  'grid_longitude': 105.882355,
  'utc_offset_seconds': 25200,
  'variables': ['temperature_2m',
   'relative_humidity_2m',
   'precipitation',
   'surface_pressure',
   'wind_speed_10m',
   'wind_direction_10m']}}

# Step 5: Save integrated hourly dataset

In [17]:
INTERIM_PATH = (
    INTERIM_DIR /
    "hanoi_openmeteo_aq_weather_hourly_2022_2026.pkl"
)

df_merged.to_pickle(INTERIM_PATH)

print("Saved:", INTERIM_PATH)
print("Rows:", len(df_merged))
print(
    "Local period:",
    df_merged["timestamp_local"].min(),
    "->",
    df_merged["timestamp_local"].max(),
)

Saved: ../data/interim/hanoi_openmeteo_aq_weather_hourly_2022_2026.pkl
Rows: 35064
Local period: 2022-09-01 00:00:00+07:00 -> 2026-08-31 23:00:00+07:00


## Collection outcome

The collection stage produces one canonical **hourly local-time timeline** for exactly four years. Air-quality and weather variables are aligned on that timeline, while both local and UTC timestamps are retained.

No modeling decision is made in this notebook. Missingness, validity checks, feature engineering, and model-ready datasets are handled in the cleaning notebook.